# Colocalization: SER fallback counts (data processing)

Share of GTEx eQTL gene × tissue and UK Biobank region × trait analyses in which SuSiE-rRSS used the SER fallback (`R_reliability_flag` of the `susie_eb_mix_res` fit), with ADSP and 1000 Genomes LD (Results, SER fallback in GTEx and UK Biobank; Fig 5a,b and S11a,b titles).

Denominator: the analyses in the per-record coloc results, i.e. pairs where the in-sample or the SuSiE-rRSS fit reported at least one credible set (the same pairs as Fig 5a,b). This is not all gene × tissue or region × trait pairs.

**Output:** `figure_data/Coloc_SER_Fallback_Counts.{rds,csv}`

In [1]:
source("../../R/utils.R")

coloc_dirs <- c(
  "GTEx eQTL|ADSP"                = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization/coloc/result",
  "GTEx eQTL|1000 Genomes"        = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_colocalization_1000G/coloc/result",
  "UKB blood traits|ADSP"         = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/coloc/result",
  "UKB blood traits|1000 Genomes" = "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization_1000_genome/coloc/result"
)

In [2]:
## one row per gene x tissue (UKB: region x trait) from the SuSiE-rRSS fit (read_coloc_records() in R/utils.R)
fallback <- imap_dfr(coloc_dirs, function(d, nm) {
  read_coloc_records(d) %>%
    filter(method == "susie_eb_mix_res") %>%
    distinct(gene, tissue, .keep_all = TRUE) %>%
    summarise(
      n_pairs    = n(),
      n_flag_na  = sum(is.na(reliability_flag)),             # no diagnostics (should be 0)
      n_fallback = sum(reliability_flag %in% TRUE),
      n_artifact = sum(artifact_flag %in% TRUE),             # flagged by Q_art
      n_sensitivity = sum(sensitivity_flag %in% TRUE)        # flagged by CS log-BF attenuation
    ) %>%
    mutate(dataset = str_split_fixed(nm, "\\|", 2)[, 1], reference = str_split_fixed(nm, "\\|", 2)[, 2], .before = 1)
}) %>%
  mutate(pct_fallback = round(100 * n_fallback / (n_pairs - n_flag_na), 1),
         text = sprintf("%.1f%% (%d/%d)", pct_fallback, n_fallback, n_pairs - n_flag_na))

fallback
save_data(fallback, "Coloc_SER_Fallback_Counts")

dataset,reference,n_pairs,n_flag_na,n_fallback,n_artifact,n_sensitivity,pct_fallback,text
<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<dbl>,<chr>
GTEx eQTL,ADSP,54175,0,586,444,201,1.1,1.1% (586/54175)
GTEx eQTL,1000 Genomes,53350,0,11743,11243,604,22.0,22.0% (11743/53350)
UKB blood traits,ADSP,4504,0,152,17,145,3.4,3.4% (152/4504)
UKB blood traits,1000 Genomes,4837,0,223,59,184,4.6,4.6% (223/4837)


In [3]:
## for the Results text
with(fallback, cat(sprintf(
  "Among comparisons with a credible set in the in-sample or adjusted fit, SER fallback was used in %s of GTEx gene-tissue analyses and %s of UK Biobank region-trait analyses with ADSP LD, compared with %s and %s, respectively, with 1000 Genomes LD.\n",
  text[dataset == "GTEx eQTL" & reference == "ADSP"],         text[dataset == "UKB blood traits" & reference == "ADSP"],
  text[dataset == "GTEx eQTL" & reference == "1000 Genomes"], text[dataset == "UKB blood traits" & reference == "1000 Genomes"])))

Among comparisons with a credible set in the in-sample or adjusted fit, SER fallback was used in 1.1% (586/54175) of GTEx gene-tissue analyses and 3.4% (152/4504) of UK Biobank region-trait analyses with ADSP LD, compared with 22.0% (11743/53350) and 4.6% (223/4837), respectively, with 1000 Genomes LD.
